# Telco Customer Churn

Can we identify customers who may leave a telecom service? I used this fictional IBM sample dataset to practice cleaning, exploring customer groups, comparing classification models, and choosing a decision threshold. This is a learning project, not a deployed retention campaign.

## 1. Load and inspect

I check missing values, distinct values, and data types first. A blank string in `TotalCharges` counts as missing even though pandas may initially read the column as text. `customerID` identifies a row; it is not a useful predictor.


In [5]:
import pandas as pd
from pathlib import Path

files = list(Path("/kaggle/input").rglob("WA_Fn-UseC_-Telco-Customer-Churn.csv"))
assert files, "ยังไม่เจอไฟล์ข้อมูล — กด Add Input แล้วเพิ่ม Telco Customer Churn ก่อน"

df = pd.read_csv(files[0])

# ช่องที่ดูเหมือนว่างแต่เป็นช่องว่าง " " ก็นับเป็น missing ด้วย
check_df = df.replace(r"^\s*$", pd.NA, regex=True)

summary = pd.DataFrame({
    "missing": check_df.isna().sum(),
    "unique": check_df.nunique(dropna=True)
})
display(df.dtypes.to_frame(name="dtype"))
display(summary)
df.head()

,dtype
customerID,object
gender,object
SeniorCitizen,int64
Partner,object
Dependents,object
tenure,int64
PhoneService,object
MultipleLines,object
InternetService,object
OnlineSecurity,object


,missing,unique
customerID,0,7043
gender,0,2
SeniorCitizen,0,2
Partner,0,2
Dependents,0,2
tenure,0,73
PhoneService,0,2
MultipleLines,0,3
InternetService,0,3
OnlineSecurity,0,3


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 2. Explore possible signals

I turn Yes/No into numbers only for this correlation check. Correlation is a clue, not proof that a feature causes churn. One-hot columns such as `OnlineSecurity_No internet service` repeat information about the same customer group.


In [6]:
corr_df = df.drop(columns="customerID").copy()

corr_df["Churn"] = corr_df["Churn"].map({"No": 0, "Yes": 1})
corr_df["TotalCharges"] = pd.to_numeric(
    corr_df["TotalCharges"], errors="coerce"
)

# แปลงคอลัมน์ข้อความ เช่น Contract เป็นคอลัมน์ 0/1
corr_df = pd.get_dummies(corr_df, dtype=int)

churn_corr = corr_df.corr(numeric_only=True)["Churn"].drop("Churn")
churn_corr = churn_corr.reindex(
    churn_corr.abs().sort_values(ascending=False).index
)

display(churn_corr.to_frame(name="correlation_with_churn"))

,correlation_with_churn
Contract_Month-to-month,0.405103
tenure,-0.352229
OnlineSecurity_No,0.342637
TechSupport_No,0.337281
InternetService_Fiber optic,0.308020
Contract_Two year,-0.302253
PaymentMethod_Electronic check,0.301919
OnlineBackup_No,0.268005
DeviceProtection_No,0.252481
StreamingMovies_No internet service,-0.227890


### Contract and customer tenure

The contract table makes the pattern easier to understand than correlation alone. I also check group sizes and median tenure so a high churn rate is not read out of context.


In [12]:
contract_churn = pd.crosstab(
    df["Contract"],
    df["Churn"],
    normalize="index"
).mul(100).round(1)

contract_summary = (
    df.assign(churned=df["Churn"].eq("Yes"))
      .groupby("Contract")
      .agg(
          customers=("Churn", "size"),
          churn_rate=("churned", "mean"),
          median_tenure=("tenure", "median")
      )
)
display(contract_churn)
display(contract_summary)

Churn,No,Yes
Contract,,
Month-to-month,57.3,42.7
One year,88.7,11.3
Two year,97.2,2.8


,customers,churn_rate,median_tenure
Contract,,,
Month-to-month,3875,42.7,12.0
One year,1473,11.3,44.0
Two year,1695,2.8,64.0


In [ ]:
import matplotlib.pyplot as plt

ax = contract_churn['Yes'].sort_values(ascending=False).plot.bar(
    color=['#dc5b5b', '#f0ad4e', '#4c9a7d'], figsize=(7, 4)
)
ax.set(title='Churn rate by contract', xlabel='', ylabel='Churn rate (%)')
ax.set_ylim(0, 50)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


### What I learned from this chart

Month-to-month customers have a 42.7% churn rate, compared with 11.3% for one-year and 2.8% for two-year contracts. They also have a much shorter median tenure (12 months versus 44 and 64). Contract choice and tenure overlap, so this does not prove that changing a contract would prevent churn.


## 3. Clean `TotalCharges`

The 11 blank values all belong to customers with `tenure = 0`. Filling them with zero could be reasonable, but I chose to remove these 11 rows (about 0.16% of the data) instead of assuming their charge history. I keep the original `df` untouched and model with `clean_df`.


In [13]:
missing_total = pd.to_numeric(df["TotalCharges"], errors="coerce").isna()

display(
    df.loc[missing_total, 
           ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]
)

,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No


In [14]:
clean_df = df.copy()

clean_df["TotalCharges"] = pd.to_numeric(
    clean_df["TotalCharges"], errors="coerce"
)
clean_df = clean_df.dropna(subset=["TotalCharges"]).copy()

print("ก่อนลบ:", len(df))
print("หลังลบ:", len(clean_df))
print("TotalCharges missing:", clean_df["TotalCharges"].isna().sum())
print("TotalCharges dtype:", clean_df["TotalCharges"].dtype)

ก่อนลบ: 7043
หลังลบ: 7032
TotalCharges missing: 0
TotalCharges dtype: float64


In [15]:
print("Missing ทั้งหมด:", clean_df.isna().sum().sum())
print("ค่าติดลบ:")
print((clean_df[["tenure", "MonthlyCharges", "TotalCharges"]] < 0).sum())

Missing ทั้งหมด: 0
ค่าติดลบ:
tenure            0
MonthlyCharges    0
TotalCharges      0
dtype: int64


## 4. Split and prepare the data

This dataset has no date to support a time-based split. I use a stratified train/validation split so both groups have a similar churn rate. Scaling and one-hot encoding live inside a pipeline and are fitted on training data only. `customerID` and the target `Churn` are not model inputs.


In [16]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X = clean_df.drop(columns=["customerID", "Churn"])
y = clean_df["Churn"].map({"No": 0, "Yes": 1})

assert y.notna().all()

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

numeric_cols = X.select_dtypes(include="number").columns
category_cols = X.select_dtypes(exclude="number").columns

preprocess = ColumnTransformer([
    ("numbers", StandardScaler(), numeric_cols),
    ("categories", OneHotEncoder(handle_unknown="ignore"), category_cols)
])

print("Train:", X_train.shape)
print("Validation:", X_valid.shape)
print("Churn ใน Validation:", y_valid.mean().round(3))

Train: (5625, 19)
Validation: (1407, 19)
Churn ใน Validation: 0.266


## 5. Compare models with a baseline

The majority-class baseline shows what a model that does not identify churn looks like. I compare Logistic Regression, Random Forest, KNN, and XGBoost on the same validation rows. AUC measures ranking; F1 balances precision and recall; recall shows how many real churn cases are caught.


In [17]:
import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score, f1_score, recall_score
from xgboost import XGBClassifier

models = {
    "Baseline": DummyClassifier(strategy="prior"),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, random_state=42, n_jobs=-1
    ),
    "KNN": KNeighborsClassifier(n_neighbors=7),
    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=3,
        learning_rate=0.05,
        tree_method="hist",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )
}

results = []

for name, model in models.items():
    pipeline = Pipeline([
        ("prepare", clone(preprocess)),
        ("model", model)
    ])
    pipeline.fit(X_train, y_train)

    train_prob = pipeline.predict_proba(X_train)[:, 1]
    valid_prob = pipeline.predict_proba(X_valid)[:, 1]
    valid_pred = pipeline.predict(X_valid)

    results.append({
        "model": name,
        "train_auc": roc_auc_score(y_train, train_prob),
        "valid_auc": roc_auc_score(y_valid, valid_prob),
        "f1": f1_score(y_valid, valid_pred, zero_division=0),
        "recall": recall_score(y_valid, valid_pred, zero_division=0)
    })

    if name == "XGBoost":
        xgb_pipeline = pipeline  # เก็บไว้ใช้ตรวจต่อ

results_df = pd.DataFrame(results)
display(results_df.sort_values("valid_auc", ascending=False).round(3))

,model,train_auc,valid_auc,f1,recall
4,XGBoost,0.876,0.837,0.587,0.543
1,Logistic Regression,0.851,0.836,0.608,0.572
2,Random Forest,1.000,0.814,0.554,0.492
3,KNN,0.887,0.797,0.558,0.570
0,Baseline,0.500,0.500,0.000,0.000


### Model choice

XGBoost and Logistic Regression have almost the same validation AUC (0.837 versus 0.836). Logistic Regression has higher F1 (0.608 versus 0.587), higher recall (0.572 versus 0.543), and a smaller train/validation AUC gap. Random Forest reaches 1.000 train AUC but only 0.814 validation AUC, a warning sign for overfitting. I continue with Logistic Regression.


### Try two fewer features

I test removing `gender` and `PhoneService` as one small ablation, not because low one-variable correlation proves they are useless. The change improves F1 and recall only slightly on this one validation split, so I do not claim a reliable gain.


In [19]:
drop_cols = ["gender", "PhoneService"]

X_train_less = X_train.drop(columns=drop_cols)
X_valid_less = X_valid.drop(columns=drop_cols)

num_less = X_train_less.select_dtypes(include="number").columns
cat_less = X_train_less.select_dtypes(exclude="number").columns

less_preprocess = ColumnTransformer([
    ("numbers", StandardScaler(), num_less),
    ("categories", OneHotEncoder(handle_unknown="ignore"), cat_less)
])

less_model = Pipeline([
    ("prepare", less_preprocess),
    ("model", LogisticRegression(max_iter=1000))
])

less_model.fit(X_train_less, y_train)

less_prob = less_model.predict_proba(X_valid_less)[:, 1]
less_pred = less_model.predict(X_valid_less)

print("Logistic เดิม:")
display(results_df.loc[
    results_df["model"] == "Logistic Regression",
    ["valid_auc", "f1", "recall"]
].round(3))

print("Logistic หลังตัด 2 คอลัมน์:")
print({
    "valid_auc": round(roc_auc_score(y_valid, less_prob), 3),
    "f1": round(f1_score(y_valid, less_pred), 3),
    "recall": round(recall_score(y_valid, less_pred), 3)
})

Logistic เดิม:


,valid_auc,f1,recall
1,0.836,0.608,0.572


Logistic หลังตัด 2 คอลัมน์:
{'valid_auc': np.float64(0.836), 'f1': 0.611, 'recall': 0.578}


## 6. Look at the actual mistakes

The confusion matrix shows false negatives: customers who left but the model predicted would stay. Those mistakes matter if the aim is to reach at-risk customers.


In [20]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_valid, less_pred)

display(pd.DataFrame(
    cm,
    index=["จริง: ไม่เลิกใช้", "จริง: เลิกใช้"],
    columns=["ทาย: ไม่เลิกใช้", "ทาย: เลิกใช้"]
))

,ทาย: ไม่เลิกใช้,ทาย: เลิกใช้
จริง: ไม่เลิกใช้,916,117
จริง: เลิกใช้,158,216


## 7. Choose a threshold for the goal

At the default threshold of 0.5, the model catches 216 of 374 churn cases and flags 117 customers who do not churn. At 0.4, it catches 256 but flags 186 who do not churn. I choose 0.4 as a *retention-focused example*: fewer missed churn cases are worth more outreach in this scenario. Without real outreach costs, I cannot call it the profit-maximizing threshold.


In [21]:
rows = []

for threshold in [0.3, 0.4, 0.5, 0.6]:
    prediction = (less_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_valid, prediction).ravel()

    rows.append({
        "threshold": threshold,
        "จับคนเลิกใช้ได้": tp,
        "พลาดคนเลิกใช้": fn,
        "ติดต่อเกิน": fp
    })

display(pd.DataFrame(rows))

,threshold,จับคนเลิกใช้ได้,พลาดคนเลิกใช้,ติดต่อเกิน
0,0.3,283,91,271
1,0.4,256,118,186
2,0.5,216,158,117
3,0.6,146,228,69


## 8. Takeaway and limits

The main actionable *hypothesis* is to pay attention to newer month-to-month customers. At threshold 0.4 the model catches 256/374 churn cases (68.4% recall), while 186 non-churn customers are flagged too. This is a trade-off, not an observed business improvement.

I used the same validation set to compare models, features, and thresholds, so these numbers may be optimistic. The next improvement would be repeated validation and a separate untouched test set before making a stronger performance claim. This dataset is fictional and contains no intervention or cost data, so it cannot show that an outreach campaign would actually reduce churn or make money.
